# Generate Monthly Power BI Snapshot

## Business Context

This notebook generates a synthetic dataset representing the monthly "Debt Held" export used within an insurance operational process to identify outstanding customer refunds.

The dataset has been designed to simulate realistic business scenarios while ensuring that no real customer or company data is included.

---

## Objectives

The notebook performs the following tasks:

- Generates realistic monthly operational snapshots
- Simulates insurance refund cases using documented business rules
- Creates consistent synthetic customer and policy data
- Produces CSV files for use within the Power BI dashboard

---

## Technologies

- Python
- Pandas
- NumPy
- Jupyter Notebook

---

## Output

The notebook exports monthly CSV snapshot files that are later combined and analysed in Power BI.


In [67]:
# Import libraries

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random 

## Configuration

This section defines the business rules used throughout the synthetic data generation process.

Values such as department distributions, cancellation rates, refund types and case volumes are configurable, making it easy to adjust the characteristics of the generated dataset without changing the generation logic.

In [ ]:
# ======================================
# Configuration
# ======================================

SNAPSHOT_MONTH = "2026-01"
SNAPSHOT_DATE = pd.Timestamp("2026-01-31")

START_MONTH = "2025-01"
NUMBER_OF_MONTHS = 18

MINIMUM_CASES_PER_MONTH = 120
MAXIMUM_CASES_PER_MONTH = 200
NUMBER_OF_CLIENTS = 200

CANCELLATION_STATUSES = {
    "Cancelled": 70,
    "Void": 30,
}

DEPARTMENTS = {
    "Retentions": 30,
    "Customer Service": 25,
    "Claims": 10,
    "Complaints": 5,
}

DEPARTMENT_AGENTS = {
    "Retentions": [f"RET{i:03d}" for i in range(1, 31)],
    "Customer Service": [f"CS{i:03d}" for i in range(1, 26)],
    "Claims": [f"CLM{i:03d}" for i in range(1, 11)],
    "Complaints": [f"CMP{i:03d}" for i in range(1, 6)],
}

DEBT_HELD_AGENTS = ["DH001", "DH002"]

REFUND_TYPES = {
    "Death of Pet": 55,
    "Cancellation Before Premium Due": 25,
    "Cancellation from Renewal (Void)": 20,
}

CUSTOMER_CONTACT_RATES = {
    "Agent Forgot": 0.45,
    "Waiting for Information": 0.30,
    "Refund Mailbox Delay": 0.25,
    "Payment Date Misunderstood": 0.10,
}

In [69]:
# ======================================
# Business Timeline
# ======================================

BUSINESS_EVENTS = {
    "training_start": pd.Timestamp("2025-07-01"),
    "new_starters": pd.Timestamp("2026-03-01"),
}

## Reproducibility

A fixed random seed is used so that the same synthetic dataset can be regenerated whenever required. This ensures the analysis remains consistent and repeatable.

In [70]:
# Set a random seed so our data is reproducible

random.seed(42)
np.random.seed(42)

print("Random seed set successfully!")

Random seed set successfully!


## Data Generation Functions

The following functions generate individual components of a synthetic Debt Held case. Keeping them separate makes the code easier to read, test and maintain.

In [71]:
# Function to generate a policy number

def generate_policy_number(number):
    """
    Generate a synthetic policy number in the format PET000001.
    """

    return f"PET{number:06d}"

In [72]:
def generate_cancellation_status(refund_type):
    """
    Determine cancellation status based on refund type.
    """

    if refund_type == "Cancellation from Renewal (Void)":
        return "Void"

    if refund_type == "Cancellation Before Premium Due":
        return "Cancelled"

    # Death of Pet can be either status
    return random.choices(
        population=["Cancelled", "Void"],
        weights=[70, 30],
        k=1,
    )[0]


def generate_cancelling_department():
    """
    Select a cancelling department using the documented distribution.
    """

    return random.choices(
        population=list(DEPARTMENTS.keys()),
        weights=list(DEPARTMENTS.values()),
        k=1,
    )[0]

In [73]:
# Function to generate a case ID

def generate_case_id(number):
    """
    Generate a synthetic Debt Held case ID.
    """

    return f"DH{number:06d}"

In [74]:
# Function to generate a client number

def generate_client_number(number):
    """
    Generate a synthetic client number.
    """

    return f"CL{number:06d}"

## Business Simulation Layer

These functions control how business conditions change over time, including training effects, new starters and seasonal workload.

In [75]:
def get_root_cause_weights(snapshot_date):
    """
    Return root-cause weights based on the business timeline
    and seasonal mailbox workload.
    """

    # Before training
    if snapshot_date < BUSINESS_EVENTS["training_start"]:
        weights = {
            "Payment Date Misunderstood": 55,
            "Agent Forgot": 20,
            "Waiting for Information": 15,
            "Refund Mailbox Delay": 10,
        }

    # After training but before new starters
    elif snapshot_date < BUSINESS_EVENTS["new_starters"]:
        weights = {
            "Payment Date Misunderstood": 40,
            "Agent Forgot": 30,
            "Waiting for Information": 20,
            "Refund Mailbox Delay": 10,
        }

    # After new starters
    else:
        weights = {
            "Payment Date Misunderstood": 45,
            "Agent Forgot": 28,
            "Waiting for Information": 17,
            "Refund Mailbox Delay": 10,
        }

    # Mailbox delays increase during December and January
    if snapshot_date.month in [12, 1]:
        weights["Refund Mailbox Delay"] += 8
        weights["Payment Date Misunderstood"] -= 8

    return weights

## Generate Complete Case

This function creates one complete synthetic Debt Held case using the documented business rules.

In [76]:
# Generate a realistic outstanding amount

def generate_outstanding_amount():

    chance = random.random()

    if chance < 0.15:
        return round(random.uniform(0.50, 10.00), 2)

    elif chance < 0.60:
        return round(random.uniform(10.01, 50.00), 2)

    elif chance < 0.90:
        return round(random.uniform(50.01, 100.00), 2)

    else:
        return round(random.uniform(100.01, 300.00), 2)

## Rule-Based Business Functions

These functions apply the documented relationships between refund type, root cause, customer contact and outcome.

In [77]:
def generate_refund_type():
    """
    Select a refund type using the documented business distribution.
    """

    return random.choices(
        population=list(REFUND_TYPES.keys()),
        weights=list(REFUND_TYPES.values()),
        k=1,
    )[0]

In [78]:
def generate_root_cause(refund_type, snapshot_date):
    """
    Generate a root cause using refund-type relationships,
    business events and seasonal workload.
    """

    weights = get_root_cause_weights(snapshot_date)

    # Adjust the general monthly weights to reflect
    # relationships between refund type and root cause
    if refund_type == "Death of Pet":
        weights["Payment Date Misunderstood"] += 10
        weights["Agent Forgot"] -= 5
        weights["Waiting for Information"] -= 5

    elif refund_type == "Cancellation from Renewal (Void)":
        weights["Agent Forgot"] += 15
        weights["Payment Date Misunderstood"] -= 10
        weights["Waiting for Information"] -= 5

    return random.choices(
        population=list(weights.keys()),
        weights=list(weights.values()),
        k=1,
    )[0]

In [79]:
def generate_customer_contact(root_cause):
    """
    Determine whether the customer contacted the business.
    """

    probability = CUSTOMER_CONTACT_RATES[root_cause]

    return "Yes" if random.random() < probability else "No"

In [80]:
def generate_outcome(refund_type):
    """
    Selects the case outcome based on the refund type.
    """

    if refund_type == "Death of Pet":
        return random.choices(
            ["Actioned", "No Action", "Raised"],
            weights=[95, 4, 1],
            k=1
        )[0]

    elif refund_type == "Cancellation Before Premium Due":
        return random.choices(
            ["Actioned", "No Action", "Raised"],
            weights=[65, 32, 3],
            k=1
        )[0]

    else:  # Cancellation from Renewal (Void)
        return random.choices(
            ["Actioned", "No Action", "Raised"],
            weights=[70, 28, 2],
            k=1
        )[0]

In [ ]:
def generate_case(case_number, snapshot_date):
    """
    Generate one complete synthetic Debt Held case.
    """

    # Unique identifiers
    case_id = generate_case_id(case_number)
    policy_number = generate_policy_number(case_number)

    client_number = generate_client_number(
        random.randint(1, NUMBER_OF_CLIENTS)
    )

    # Case details
    outstanding_amount = generate_outstanding_amount()
    refund_type = generate_refund_type()
    cancellation_status = generate_cancellation_status(refund_type)
    cancelling_department = generate_cancelling_department()




    # Choose an agent belonging to the cancelling department
    cancelling_agent = random.choice(
        DEPARTMENT_AGENTS[cancelling_department]
    )

    # Choose one of the trained Debt Held agents
    agent_working = random.choice(DEBT_HELD_AGENTS)

    # Refund investigation
    refund_type = generate_refund_type()
    cancellation_status = generate_cancellation_status(refund_type)
    root_cause = generate_root_cause(
    refund_type,
    snapshot_date,
)
    customer_contacted = generate_customer_contact(root_cause)
    outcome = generate_outcome(refund_type)

    return {
        "Case ID": case_id,
        "Policy Number": policy_number,
        "Client Number": client_number,
        "Snapshot Date": snapshot_date,
        "Outstanding Amount": outstanding_amount,
        "Cancellation Status": cancellation_status,
        "Cancelling Department": cancelling_department,
        "Cancelling Agent": cancelling_agent,
        "Agent Working": agent_working,
        "Refund Type": refund_type,
        "Root Cause": root_cause,
        "Customer Contacted": customer_contacted,
        "Outcome": outcome,
    }

## Generate 18 month Snapshot

Generate the complete synthetic monthly dataset using the configured number of cases.


In [82]:
# Generate multiple monthly snapshots

all_monthly_data = []

month_dates = pd.date_range(
    start=START_MONTH,
    periods=NUMBER_OF_MONTHS,
    freq="ME",
)

case_number = 1

for snapshot_date in month_dates:

    # December and January are busier months
    if snapshot_date.month in [12, 1]:
        monthly_case_count = random.randint(175, 200)

    # Other months have normal workload variation
    else:
        monthly_case_count = random.randint(
            MINIMUM_CASES_PER_MONTH,
            180,
        )

    monthly_cases = []

    for _ in range(monthly_case_count):
        monthly_cases.append(
            generate_case(
                case_number,
                snapshot_date,
            )
        )

        case_number += 1

    monthly_df = pd.DataFrame(monthly_cases)

    file_name = (
        f"powerbi_snapshot_"
        f"{snapshot_date.strftime('%Y_%m')}.csv"
    )

    output_path = f"../data/raw/{file_name}"

    monthly_df.to_csv(
        output_path,
        index=False,
    )

    all_monthly_data.append(monthly_df)

    print(
        f"Saved {file_name}: "
        f"{len(monthly_df)} cases"
    )

# Combine every monthly snapshot into one DataFrame
snapshot_df = pd.concat(
    all_monthly_data,
    ignore_index=True,
)

snapshot_df.head()

Saved powerbi_snapshot_2025_01.csv: 195 cases
Saved powerbi_snapshot_2025_02.csv: 139 cases
Saved powerbi_snapshot_2025_03.csv: 133 cases
Saved powerbi_snapshot_2025_04.csv: 177 cases
Saved powerbi_snapshot_2025_05.csv: 160 cases
Saved powerbi_snapshot_2025_06.csv: 170 cases
Saved powerbi_snapshot_2025_07.csv: 179 cases
Saved powerbi_snapshot_2025_08.csv: 142 cases
Saved powerbi_snapshot_2025_09.csv: 161 cases
Saved powerbi_snapshot_2025_10.csv: 145 cases
Saved powerbi_snapshot_2025_11.csv: 134 cases
Saved powerbi_snapshot_2025_12.csv: 175 cases
Saved powerbi_snapshot_2026_01.csv: 191 cases
Saved powerbi_snapshot_2026_02.csv: 179 cases
Saved powerbi_snapshot_2026_03.csv: 133 cases
Saved powerbi_snapshot_2026_04.csv: 144 cases
Saved powerbi_snapshot_2026_05.csv: 171 cases
Saved powerbi_snapshot_2026_06.csv: 143 cases


,Case ID,Policy Number,Client Number,Snapshot Date,Outstanding Amount,Cancellation Status,Cancelling Department,Cancelling Agent,Agent Working,Refund Type,Root Cause,Customer Contacted,Outcome
0,DH000001,PET000001,CL000029,2025-01-31,3.11,Cancelled,Customer Service,CS018,DH001,Cancellation Before Premium Due,Payment Date Misunderstood,Yes,Actioned
1,DH000002,PET000002,CL000155,2025-01-31,2.39,Cancelled,Customer Service,CS008,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,No Action
2,DH000003,PET000003,CL000179,2025-01-31,21.12,Cancelled,Retentions,RET013,DH001,Death of Pet,Payment Date Misunderstood,Yes,Actioned
3,DH000004,PET000004,CL000032,2025-01-31,175.71,Cancelled,Claims,CLM010,DH002,Cancellation Before Premium Due,Waiting for Information,Yes,Actioned
4,DH000005,PET000005,CL000075,2025-01-31,271.06,Cancelled,Retentions,RET015,DH002,Death of Pet,Agent Forgot,No,Actioned


In [83]:
snapshot_df.groupby(
    snapshot_df["Snapshot Date"].dt.to_period("M")
).size()

Snapshot Date
2025-01    195
2025-02    139
2025-03    133
2025-04    177
2025-05    160
2025-06    170
2025-07    179
2025-08    142
2025-09    161
2025-10    145
2025-11    134
2025-12    175
2026-01    191
2026-02    179
2026-03    133
2026-04    144
2026-05    171
2026-06    143
Freq: M, dtype: int64

In [84]:
test_case = generate_case(
    case_number=1,
    snapshot_date=pd.Timestamp("2025-12-31"),
)

test_case

{'Case ID': 'DH000001',
 'Policy Number': 'PET000001',
 'Client Number': 'CL000053',
 'Snapshot Date': Timestamp('2025-12-31 00:00:00'),
 'Outstanding Amount': 65.92,
 'Cancellation Status': 'Void',
 'Cancelling Department': 'Retentions',
 'Cancelling Agent': 'RET021',
 'Agent Working': 'DH002',
 'Refund Type': 'Death of Pet',
 'Root Cause': 'Waiting for Information',
 'Customer Contacted': 'No',
 'Outcome': 'Actioned'}

In [85]:
get_root_cause_weights(pd.Timestamp("2025-06-30"))

{'Payment Date Misunderstood': 55,
 'Agent Forgot': 20,
 'Waiting for Information': 15,
 'Refund Mailbox Delay': 10}

In [86]:
get_root_cause_weights(pd.Timestamp("2025-12-31"))

{'Payment Date Misunderstood': 32,
 'Agent Forgot': 30,
 'Waiting for Information': 20,
 'Refund Mailbox Delay': 18}

In [87]:
root_cause_by_month = pd.crosstab(
    snapshot_df["Snapshot Date"].dt.to_period("M"),
    snapshot_df["Root Cause"],
    normalize="index",
).mul(100).round(1)

root_cause_by_month

Root Cause,Agent Forgot,Payment Date Misunderstood,Refund Mailbox Delay,Waiting for Information
Snapshot Date,,,,
2025-01,22.1,48.2,17.4,12.3
2025-02,23.7,60.4,5.8,10.1
2025-03,21.8,57.9,11.3,9.0
2025-04,26.0,52.5,10.2,11.3
2025-05,21.2,58.8,13.1,6.9
2025-06,18.8,59.4,9.4,12.4
2025-07,29.1,46.4,9.5,15.1
2025-08,32.4,39.4,10.6,17.6
2025-09,29.2,43.5,8.1,19.3
